# Week 5, Notebook 5: Computer Vision in the Caribbean

**Seeing what a CNN learns, and where vision helps the region.**

*Caribbean AI - Adrian Dunkley*

---

### What you will build
- A window **inside** a trained CNN: the filters it learned and the feature maps
  it makes.
- An **occlusion test** that reveals which part of a flag the network relies on.
- A grounded tour of **real computer-vision uses across the Caribbean**, plus the
  limits and ethics you must keep in mind.

### The big idea
A CNN can feel like a black box. It is not. In this final notebook we open it up
and *look*: the little filters it invented, the patterns those filters detect,
and the exact pixels that change its mind. Understanding *why* a model decides is
as important as its accuracy, especially when vision is used for real decisions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

# Load flags and train a small CNN quickly (same idea as Notebook 3).
data = np.load("data/caribbean_flags.npz", allow_pickle=True)
X = data["X"].astype("float32") / 255.0
y = data["y"]
class_names = list(data["class_names"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)

tf.random.set_seed(0)
model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(16, 3, activation="relu", name="conv1"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu", name="conv2"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(6, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=8, batch_size=32, verbose=0)
print("Trained. Test accuracy:", round(model.evaluate(X_test, y_test, verbose=0)[1], 3))

### Step 1: The filters the network invented

The first conv layer learned 16 small 3x3 filters, one per colour channel. In
Notebook 2 we *designed* an edge filter by hand; here the network **learned** its
own from the flags. Let us look at them. They are tiny and abstract, but you can
often spot filters tuned to certain colours or edges.

In [ ]:
conv1 = model.get_layer("conv1")
filters = conv1.get_weights()[0]        # shape (3, 3, 3, 16): 16 filters
# normalise for display
f = filters - filters.min()
f = f / f.max()

fig, axes = plt.subplots(2, 8, figsize=(13, 3.3))
for i, ax in enumerate(axes.flat):
    ax.imshow(f[:, :, :, i])            # show each 3x3x3 filter as a colour patch
    ax.set_title(f"filter {i}", fontsize=7); ax.axis("off")
plt.suptitle("The 16 filters the first conv layer learned")
plt.show()

### Step 2: Feature maps - what the filters see in a flag

Feed one flag in and capture what each first-layer filter lights up on. These
**feature maps** are the "edges and colours found" version of the image. Bright
areas mean "this filter fired strongly here".

In [ ]:
# Build a mini-model that outputs the conv1 activations.
activation_model = keras.Model(inputs=model.inputs,
                               outputs=model.get_layer("conv1").output)

sample_idx = np.where(y_test == class_names.index("Jamaica"))[0][0]
sample = X_test[sample_idx][None, ...]
maps = activation_model.predict(sample, verbose=0)[0]   # (30, 30, 16)

fig, axes = plt.subplots(2, 9, figsize=(15, 3.6))
axes[0, 0].imshow(sample[0]); axes[0, 0].set_title("input flag", fontsize=8)
axes[0, 0].axis("off")
for i in range(16):
    ax = axes[(i + 1) // 9, (i + 1) % 9]
    ax.imshow(maps[:, :, i], cmap="viridis")
    ax.set_title(f"map {i}", fontsize=7); ax.axis("off")
plt.suptitle("What each learned filter detects in the Jamaica flag")
plt.show()

### Step 3: Which pixels matter? An occlusion test

Here is a simple way to ask the model *"what are you looking at?"*. We slide a
small grey square over the flag, and each time we check how much the model's
confidence in the correct answer drops. Where hiding the flag hurts the most,
that is where the model was looking. The result is a **heat map** of importance.

In [ ]:
def occlusion_heatmap(image, true_label, patch=6, step=3):
    base_conf = model.predict(image[None, ...], verbose=0)[0][true_label]
    h, w = image.shape[:2]
    heat = np.zeros(((h - patch) // step + 1, (w - patch) // step + 1))
    for a, i in enumerate(range(0, h - patch + 1, step)):
        for b, j in enumerate(range(0, w - patch + 1, step)):
            occluded = image.copy()
            occluded[i:i+patch, j:j+patch] = 0.5      # grey out a patch
            conf = model.predict(occluded[None, ...], verbose=0)[0][true_label]
            heat[a, b] = base_conf - conf             # drop in confidence
    return heat

img = X_test[sample_idx]
heat = occlusion_heatmap(img, y_test[sample_idx])

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img); axes[0].set_title("Jamaica flag"); axes[0].axis("off")
axes[1].imshow(heat, cmap="hot"); axes[1].set_title("what the model relies on"); axes[1].axis("off")
plt.show()
print("Bright areas = hiding them most confused the model, so it looks there.")

### Step 4: Where computer vision helps the Caribbean

The exact tools you built this week power real projects across the region. A few:

- **Hurricane and storm tracking.** CNNs read satellite imagery to spot and
  follow systems threatening Jamaica, Haiti, Cuba, the Bahamas, and beyond, just
  like our Notebook 4 detector, but at full scale.
- **Sargassum seaweed detection.** Vision models scan coastal and satellite
  images to predict where sargassum will wash up, helping beaches from Barbados
  to Tobago prepare.
- **Coral reef health.** Divers' photos and drone footage are classified to
  monitor bleaching on reefs around the Bahamas, Belize, and the Grenadines.
- **Crop and banana disease.** Farmers in Saint Lucia, Dominica, and Jamaica
  photograph leaves; a CNN flags disease early, protecting harvests.
- **Road safety and traffic.** Cameras count vehicles and spot hazards in
  Kingston, Port of Spain, and other busy Caribbean cities.
- **Tourism and heritage.** Vision helps catalogue reefs, wildlife, and historic
  sites that draw visitors across the islands.

### Step 5: The limits and the ethics

A CNN is powerful but not wise. Keep these in mind, always:

- **It only knows what it was shown.** A flag model trained on six flags will
  confidently mislabel a seventh. Real models need broad, representative data
  from across the Caribbean, not one island's images.
- **Bias in, bias out.** If the training photos over-represent some places,
  skin tones, or conditions, the model works worse for the rest. Fairness starts
  with the data.
- **Confidence is not correctness.** A model can be 99% sure and wrong. For
  high-stakes uses (health, safety, security), a human stays in charge.
- **Privacy matters.** Cameras and image data touch real people. Collect and use
  them with consent and care.

Used thoughtfully, computer vision is a genuine tool for Caribbean problems, from
weathering storms to protecting reefs and crops.

### What comes next: transfer learning

You trained small CNNs from scratch. Professionals often start from a big model
already trained on millions of images (like MobileNet or ResNet) and **fine-tune**
it on their own smaller dataset. This is **transfer learning**, and it lets a
Caribbean team build a strong reef or crop classifier from only a few hundred
local photos. It is the natural next step after this course.

### What you learned

- You can **open up a CNN**: view its learned **filters**, its **feature maps**,
  and, with an **occlusion test**, the pixels it relies on.
- Computer vision already serves the Caribbean, from **hurricane tracking** to
  **sargassum**, **reefs**, and **crops**.
- Models are only as good and as fair as their **data**, confidence is not
  correctness, and humans must stay in charge of high-stakes decisions.
- **Transfer learning** is your path to strong models from small local datasets.

### Try it yourself
1. Run the occlusion test on a Trinidad and Tobago flag. Does the model look at
   the diagonal stripe?
2. Pick the filter with the most colourful feature map and describe what it seems
   to detect.
3. Brainstorm a Caribbean problem you could solve with a CNN, and list what
   training images you would need to do it fairly.